# 🟢 **Generate Test**

In [2]:
# -*- coding: utf-8 -*-
"""
Complete IndependentARK test + Persian text layers.

Adds:
1) Input normalization: non-half-spaced forms -> canonical half-spaced forms.
2) Tone detection from the USER INPUT.
3) Tone detection from a HYPOTHETICAL MODEL RESPONSE.
4) Tone consistency test between user input and hypothetical response.
5) Formal -> casual conversion according to casual.json.
6) Tokenizer tests, next-token probability, greedy decoding, top-k sampling.
7) Layered generation: normalize input -> detect input tone -> generate -> detect
   generated tone -> optionally convert generated output to the user's tone.
"""

import os
import json
import math
import re
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F
from tokenizers import Tokenizer


# ============================================================
# CONFIG
# ============================================================

DATA_ROOT = "/data"

BEST_MODEL_PATH = os.path.join(
    DATA_ROOT,
    "logs",
    "Alphabet_NLP_Dataset_20260906_063836",
    "best_model.pt",
)

TOKENIZER_PATH = os.path.join(
    DATA_ROOT,
    "tokenizer",
    "bpe-tokenizer_alphabet_nlp_dataset.json",
)

TEXT_LAYERS_DIR = os.path.join(
    DATA_ROOT,
    "raw",
    "persian_text_layers",
)

HALFSPACE_PATH = os.path.join(
    TEXT_LAYERS_DIR,
    "halfspace.json",
)

CASUAL_PATH = os.path.join(
    TEXT_LAYERS_DIR,
    "casual.json",
)

TEST_DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=== MODAL PATH DEBUG ===")

for path in ["/data", "/raw", "/mnt", "/workspace"]:
    print(f"\n{path}:")
    if os.path.exists(path):
        print(os.listdir(path)[:30])
    else:
        print("NOT FOUND")


# ============================================================
# TEXT LAYERS
# ============================================================

class PersianTextLayers:
    """
    Non-destructive text layer.

    - normalize_input(): only replaces forms explicitly present in
      halfspace.json.
    - detect_tone(): heuristic classifier for formal/casual/neutral.
    - casualize_output(): applies formal_to_casual mapping.
    """

    def __init__(self, halfspace_path, casual_path):
        self.halfspace_path = halfspace_path
        self.casual_path = casual_path

        self._load()

    def _load(self):
        with open(self.halfspace_path, "r", encoding="utf-8") as f:
            halfspace_data = json.load(f)

        with open(self.casual_path, "r", encoding="utf-8") as f:
            casual_data = json.load(f)

        self.normalization = halfspace_data.get(
            "normalization",
            {}
        )

        self.correct_forms = set(
            halfspace_data.get(
                "correct_forms",
                []
            )
        )

        self.formal_to_casual = casual_data.get(
            "formal_to_casual",
            {}
        )

        self.casual_to_formal = {
            casual: formal
            for formal, casual
            in self.formal_to_casual.items()
        }

        # Longest first prevents a short rule from firing inside a longer one.
        self._normalization_items = sorted(
            self.normalization.items(),
            key=lambda x: len(x[0]),
            reverse=True,
        )

        self._formal_items = sorted(
            self.formal_to_casual.items(),
            key=lambda x: len(x[0]),
            reverse=True,
        )

        # Weighted tone markers.
        self.formal_markers = {
            "می‌توانی": 2.0,
            "می‌توانم": 2.0,
            "می‌توانید": 2.0,
            "می‌شود": 1.5,
            "می‌گویید": 2.0,
            "می‌خواهی": 2.0,
            "می‌خواهم": 2.0,
            "می‌آیی": 2.0,
            "می‌رود": 1.5,
            "دارید": 1.5,
            "هستید": 1.5,
            "نیستید": 1.5,
            "شما": 1.0,
            "آن": 0.5,
            "آنجا": 0.5,
            "چطور": 1.0,
            "چه": 0.25,
            "کدام": 1.0,
            "زیرا": 1.5,
            "متشکرم": 2.0,
            "بله": 1.0,
            "خیر": 1.0,
        }

        self.casual_markers = {
            "می‌تونی": 2.0,
            "می‌تونم": 2.0,
            "می‌تونید": 2.0,
            "می‌شه": 1.5,
            "می‌گید": 2.0,
            "می‌خوای": 2.0,
            "می‌خوام": 2.0,
            "میای": 2.0,
            "میره": 1.5,
            "دارین": 1.5,
            "هستین": 1.5,
            "نیستین": 1.5,
            "تو": 1.0,
            "اون": 0.5,
            "اونجا": 0.5,
            "چطوری": 1.0,
            "چی": 0.25,
            "کدوم": 1.0,
            "چون": 1.5,
            "مرسی": 2.0,
            "آره": 1.0,
            "نه": 1.0,
        }

    @staticmethod
    def _replace_longest_first(text, items):
        for old, new in items:
            text = text.replace(old, new)
        return text

    def normalize_input(self, text):
        """
        Convert only explicitly registered incorrect spellings to
        canonical preferred spellings.
        """
        return self._replace_longest_first(
            text,
            self._normalization_items,
        )

    def detect_tone(self, text):
        """
        Returns:
            formal / casual / neutral
        """
        if not text or not text.strip():
            return "neutral"

        formal_score = 0.0
        casual_score = 0.0

        # Normalize only the spelling layer first so the tone classifier
        # sees the canonical forms used by the dictionaries.
        normalized = self.normalize_input(text)

        for marker, weight in self.formal_markers.items():
            formal_score += normalized.count(marker) * weight

        for marker, weight in self.casual_markers.items():
            casual_score += normalized.count(marker) * weight

        # Very short ambiguous markers should not dominate the result.
        ambiguous = {
            "تو", "اون", "چی", "نه",
            "چه", "آن", "بله", "خیر",
        }

        for marker in ambiguous:
            if marker in self.formal_markers:
                formal_score -= normalized.count(marker) * self.formal_markers[marker]
            if marker in self.casual_markers:
                casual_score -= normalized.count(marker) * self.casual_markers[marker]

        # Re-add ambiguous markers with low weight.
        low_weight = {
            "تو": 0.25,
            "اون": 0.25,
            "چی": 0.25,
            "نه": 0.25,
            "چه": 0.15,
            "آن": 0.15,
            "بله": 0.25,
            "خیر": 0.25,
        }

        for marker, weight in low_weight.items():
            if marker in self.formal_markers:
                formal_score += normalized.count(marker) * weight
            if marker in self.casual_markers:
                casual_score += normalized.count(marker) * weight

        if casual_score > formal_score and casual_score >= 1.0:
            tone = "casual"
        elif formal_score > casual_score and formal_score >= 1.0:
            tone = "formal"
        else:
            tone = "neutral"

        return {
            "tone": tone,
            "formal_score": round(formal_score, 3),
            "casual_score": round(casual_score, 3),
        }

    def casualize_output(self, text):
        return self._replace_longest_first(
            text,
            self._formal_items,
        )

    def formalize_output(self, text):
        items = sorted(
            self.casual_to_formal.items(),
            key=lambda x: len(x[0]),
            reverse=True,
        )
        return self._replace_longest_first(
            text,
            items,
        )

    def preprocess_prompt(self, prompt):
        normalized = self.normalize_input(prompt)
        tone = self.detect_tone(normalized)

        return {
            "original": prompt,
            "normalized": normalized,
            "tone": tone,
        }

    def postprocess_response(
        self,
        response,
        target_tone,
        auto_convert=True,
    ):
        detected = self.detect_tone(response)

        if (
            auto_convert
            and target_tone == "casual"
            and detected["tone"] != "casual"
        ):
            converted = self.casualize_output(response)
        elif (
            auto_convert
            and target_tone == "formal"
            and detected["tone"] == "casual"
        ):
            converted = self.formalize_output(response)
        else:
            converted = response

        return {
            "original": response,
            "detected_tone": detected,
            "converted": converted,
            "converted_tone": self.detect_tone(converted),
        }


# ============================================================
# LOAD TEXT LAYERS
# ============================================================

print("=" * 100)
print("📝 PERSIAN TEXT LAYERS")
print("=" * 100)

for required_path in [
    HALFSPACE_PATH,
    CASUAL_PATH,
]:
    if not os.path.exists(required_path):
        raise FileNotFoundError(
            f"❌ Text layer file not found:\n{required_path}"
        )

text_layers = PersianTextLayers(
    HALFSPACE_PATH,
    CASUAL_PATH,
)

print(f"✅ halfspace.json loaded: {HALFSPACE_PATH}")
print(f"   Normalization rules: {len(text_layers.normalization):,}")
print(f"   Correct forms:       {len(text_layers.correct_forms):,}")
print(f"✅ casual.json loaded:   {CASUAL_PATH}")
print(
    f"   Formal→casual rules: "
    f"{len(text_layers.formal_to_casual):,}"
)


# ============================================================
# TEXT LAYER TEST 1:
# NON-HALF-SPACE INPUT -> HALF-SPACE
# ============================================================

print("\n")
print("=" * 100)
print("🧪 TEST 1 — INPUT NORMALIZATION")
print("=" * 100)

normalization_tests = [
    "بالغ بر",
    "حمل و نقل",
    "زنده گی",
    "می رود",
    "می شود",
    "می تواند",
    "به کار گیری",
]

normalization_passed = 0

for original in normalization_tests:
    normalized = text_layers.normalize_input(original)

    print("\nOriginal :")
    print(repr(original))
    print("Normalized:")
    print(repr(normalized))

    if original != normalized:
        normalization_passed += 1
        print("RESULT: ✅ CONVERTED")
    else:
        print("RESULT: ⚠️ NO REGISTERED CHANGE")

print(
    f"\nNormalization tests with a change: "
    f"{normalization_passed}/{len(normalization_tests)}"
)


# ============================================================
# TEXT LAYER TEST 2:
# USER INPUT TONE DETECTION
# ============================================================

print("\n")
print("=" * 100)
print("🧪 TEST 2 — USER INPUT TONE DETECTION")
print("=" * 100)

user_tone_tests = [
    (
        "لطفاً می‌توانید این موضوع را برای من توضیح دهید؟",
        "formal",
    ),
    (
        "می‌تونی این موضوع رو برام توضیح بدی؟",
        "casual",
    ),
    (
        "متشکرم، آیا می‌توانید بیشتر توضیح دهید؟",
        "formal",
    ),
    (
        "مرسی، می‌تونی بیشتر توضیح بدی؟",
        "casual",
    ),
]

tone_passed = 0

for user_text, expected_tone in user_tone_tests:
    result = text_layers.detect_tone(user_text)

    print("\nUser input:")
    print(user_text)
    print("Expected:", expected_tone)
    print("Detected:", result)

    if result["tone"] == expected_tone:
        tone_passed += 1
        print("RESULT: ✅ PASS")
    else:
        print("RESULT: ❌ FAIL")

print(
    f"\nUser tone tests: "
    f"{tone_passed}/{len(user_tone_tests)} passed"
)


# ============================================================
# TEXT LAYER TEST 3:
# HYPOTHETICAL MODEL RESPONSE TONE DETECTION
# ============================================================

print("\n")
print("=" * 100)
print("🧪 TEST 3 — HYPOTHETICAL MODEL RESPONSE TONE DETECTION")
print("=" * 100)

hypothetical_response_tests = [
    (
        "بله، می‌توانم این موضوع را برای شما توضیح دهم.",
        "formal",
    ),
    (
        "آره، می‌تونم این موضوع رو برات توضیح بدم.",
        "casual",
    ),
    (
        "خیر، در حال حاضر نمی‌توانم این درخواست را انجام دهم.",
        "formal",
    ),
    (
        "نه، الان نمی‌تونم این کار رو انجام بدم.",
        "casual",
    ),
]

response_tone_passed = 0

for response_text, expected_tone in hypothetical_response_tests:
    result = text_layers.detect_tone(response_text)

    print("\nHypothetical model response:")
    print(response_text)
    print("Expected:", expected_tone)
    print("Detected:", result)

    if result["tone"] == expected_tone:
        response_tone_passed += 1
        print("RESULT: ✅ PASS")
    else:
        print("RESULT: ❌ FAIL")

print(
    f"\nHypothetical response tone tests: "
    f"{response_tone_passed}/{len(hypothetical_response_tests)} passed"
)


# ============================================================
# TEXT LAYER TEST 4:
# USER TONE vs MODEL RESPONSE TONE
# ============================================================

print("\n")
print("=" * 100)
print("🧪 TEST 4 — USER/RESPONSE TONE CONSISTENCY")
print("=" * 100)

tone_pair_tests = [
    (
        "می‌تونی اینو برام توضیح بدی؟",
        "بله، می‌توانم این موضوع را برای شما توضیح دهم.",
        "casual",
        "formal",
    ),
    (
        "لطفاً می‌توانید این موضوع را توضیح دهید؟",
        "بله، می‌توانم این موضوع را برای شما توضیح دهم.",
        "formal",
        "formal",
    ),
    (
        "مرسی، می‌تونی بیشتر توضیح بدی؟",
        "آره، می‌تونم بیشتر توضیح بدم.",
        "casual",
        "casual",
    ),
]

pair_passed = 0

for user_text, model_response, expected_user, expected_response in tone_pair_tests:
    user_result = text_layers.detect_tone(user_text)
    response_result = text_layers.detect_tone(model_response)

    consistent = (
        user_result["tone"] == response_result["tone"]
    )

    print("\nUser:")
    print(user_text)
    print("User tone:", user_result)

    print("\nHypothetical model response:")
    print(model_response)
    print("Response tone:", response_result)

    print(
        "\nExpected:",
        expected_user,
        "->",
        expected_response,
    )

    if (
        user_result["tone"] == expected_user
        and response_result["tone"] == expected_response
    ):
        pair_passed += 1
        print("Detection: ✅ PASS")
    else:
        print("Detection: ❌ FAIL")

    print(
        "Tone consistency:",
        "✅ MATCH" if consistent else "⚠️ MISMATCH",
    )

print(
    f"\nUser/response pair tests: "
    f"{pair_passed}/{len(tone_pair_tests)} passed"
)


# ============================================================
# TEXT LAYER TEST 5:
# FORMAL RESPONSE -> CASUAL RESPONSE
# ============================================================

print("\n")
print("=" * 100)
print("🧪 TEST 5 — FORMAL -> CASUAL RESPONSE CONVERSION")
print("=" * 100)

conversion_tests = [
    (
        "بله، می‌توانم این موضوع را برای شما توضیح دهم.",
        "آره، می‌تونم این موضوع رو برات توضیح بدم.",
    ),
    (
        "متشکرم. اگر می‌خواهید، می‌توانم آن را بیشتر توضیح دهم.",
        "مرسی. اگه می‌خوای، می‌تونم اون رو بیشتر توضیح بدم.",
    ),
]

conversion_passed = 0

for formal_text, expected_casual in conversion_tests:
    converted = text_layers.casualize_output(formal_text)

    print("\nFormal:")
    print(formal_text)
    print("Converted:")
    print(converted)
    print("Expected:")
    print(expected_casual)

    if converted == expected_casual:
        conversion_passed += 1
        print("RESULT: ✅ PASS")
    else:
        print("RESULT: ⚠️ PARTIAL / CHECK MAPPING")

print(
    f"\nFormal→casual exact tests: "
    f"{conversion_passed}/{len(conversion_tests)} passed"
)


# ============================================================
# TEXT LAYER TEST 6:
# FULL PIPELINE WITH USER INPUT + HYPOTHETICAL RESPONSE
# ============================================================

print("\n")
print("=" * 100)
print("🧪 TEST 6 — FULL TEXT PIPELINE")
print("=" * 100)

pipeline_user = (
    "می تونی حمل و نقل رو برام توضیح بدی؟"
)

pipeline_response = (
    "بله، می‌توانم حمل‌ونقل را برای شما توضیح دهم."
)

preprocessed = text_layers.preprocess_prompt(
    pipeline_user
)

processed_response = text_layers.postprocess_response(
    pipeline_response,
    target_tone=preprocessed["tone"]["tone"],
    auto_convert=True,
)

print("\n[User original]")
print(preprocessed["original"])

print("\n[User normalized]")
print(preprocessed["normalized"])

print("\n[User detected tone]")
print(preprocessed["tone"])

print("\n[Model hypothetical response]")
print(processed_response["original"])

print("\n[Model detected tone]")
print(processed_response["detected_tone"])

print("\n[Final response after tone conversion]")
print(processed_response["converted"])

print("\n[Final response tone]")
print(processed_response["converted_tone"])


# ============================================================
# START MODEL TEST
# ============================================================

print("\n")
print("=" * 100)
print("🧪 INDEPENDENT BEST MODEL TEST")
print("=" * 100)

print(f"Device: {TEST_DEVICE}")
print(f"Tokenizer: {TOKENIZER_PATH}")
print(f"Model: {BEST_MODEL_PATH}")


# ============================================================
# FILE CHECK
# ============================================================

if not os.path.exists(TOKENIZER_PATH):
    raise FileNotFoundError(
        f"❌ Tokenizer not found:\n{TOKENIZER_PATH}"
    )

if not os.path.exists(BEST_MODEL_PATH):
    raise FileNotFoundError(
        f"❌ Best model not found:\n{BEST_MODEL_PATH}"
    )

print("\n✅ Tokenizer file found.")
print("✅ Best model file found.")


# ============================================================
# LOAD TOKENIZER
# ============================================================

test_tokenizer = Tokenizer.from_file(
    TOKENIZER_PATH
)

print("\n✅ Tokenizer loaded successfully.")
print(
    f"Vocabulary size: "
    f"{test_tokenizer.get_vocab_size():,}"
)


# ============================================================
# MULTI-HEAD ATTENTION
# ============================================================

class IndependentMultiHeadAttention(nn.Module):

    def __init__(
        self,
        n_embd,
        n_head,
        dropout_rate
    ):
        super().__init__()

        if n_embd % n_head != 0:
            raise ValueError(
                "n_embd must be divisible by n_head."
            )

        self.n_embd = n_embd
        self.n_head = n_head
        self.head_size = n_embd // n_head

        self.qkv_proj = nn.Linear(
            n_embd,
            3 * n_embd,
            bias=False
        )

        self.c_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.c_proj.residual = True

    def forward(self, x):

        B, T, C = x.shape

        qkv = (
            self.qkv_proj(x)
            .view(
                B,
                T,
                3,
                self.n_head,
                self.head_size
            )
            .permute(
                2,
                0,
                3,
                1,
                4
            )
        )

        q, k, v = qkv.unbind(0)

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True
        )

        y = (
            y.transpose(1, 2)
            .contiguous()
            .view(
                B,
                T,
                C
            )
        )

        return self.c_proj(y)


# ============================================================
# FEED FORWARD
# ============================================================

class IndependentFeedForward(nn.Module):

    def __init__(
        self,
        n_embd,
        f_expnd,
        dropout_rate
    ):
        super().__init__()

        hidden_size = int(
            f_expnd * n_embd
        )

        self.up_proj = nn.Linear(
            n_embd,
            hidden_size,
            bias=False
        )

        self.down_proj = nn.Linear(
            hidden_size,
            n_embd,
            bias=False
        )

        self.down_proj.residual = True

        self.mlp_dropout = nn.Dropout(
            dropout_rate
        )

    def forward(self, x):

        return self.mlp_dropout(
            self.down_proj(
                F.gelu(
                    self.up_proj(x)
                )
            )
        )


# ============================================================
# DECODER BLOCK
# ============================================================

class IndependentDecoderBlock(nn.Module):

    def __init__(
        self,
        n_embd,
        n_head,
        f_expnd,
        dropout_rate
    ):
        super().__init__()

        self.ln1 = nn.LayerNorm(
            n_embd
        )

        self.mha = IndependentMultiHeadAttention(
            n_embd,
            n_head,
            dropout_rate
        )

        self.ln2 = nn.LayerNorm(
            n_embd
        )

        self.mlp = IndependentFeedForward(
            n_embd,
            f_expnd,
            dropout_rate
        )

        self.dropout = nn.Dropout(
            dropout_rate
        )

    def forward(self, x):

        x = x + self.dropout(
            self.mha(
                self.ln1(x)
            )
        )

        x = x + self.dropout(
            self.mlp(
                self.ln2(x)
            )
        )

        return x


# ============================================================
# ARK MODEL
# ============================================================

class IndependentARK(nn.Module):

    def __init__(
        self,
        vocab_size=10000,
        max_seq_len=1024,
        n_layer=8,
        n_head=16,
        n_embd=128,
        f_expnd=4,
        dropout_rate=0.2
    ):
        super().__init__()

        self.wte = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.wpe = nn.Embedding(
            max_seq_len,
            n_embd
        )

        self.decoders = nn.ModuleList(
            [
                IndependentDecoderBlock(
                    n_embd=n_embd,
                    n_head=n_head,
                    f_expnd=f_expnd,
                    dropout_rate=dropout_rate
                )
                for _ in range(n_layer)
            ]
        )

        self.lnf = nn.LayerNorm(
            n_embd
        )

        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = self.wte.weight

    def forward(self, idx):

        B, T = idx.shape

        if T > self.wpe.num_embeddings:
            raise ValueError(
                f"Sequence length {T} exceeds "
                f"max_seq_len={self.wpe.num_embeddings}"
            )

        positions = torch.arange(
            T,
            device=idx.device
        )

        x = (
            self.wte(idx)
            +
            self.wpe(positions)
        )

        for decoder in self.decoders:
            x = decoder(x)

        x = self.lnf(x)

        return self.lm_head(x)


# ============================================================
# CREATE MODEL
# ============================================================

test_model = IndependentARK(
    vocab_size=10_000,
    max_seq_len=1024,
    n_layer=8,
    n_head=16,
    n_embd=128,
    f_expnd=4,
    dropout_rate=0.2
).to(TEST_DEVICE)

print(
    "\n✅ Independent ARK architecture created."
)


# ============================================================
# PARAMETER COUNT
# ============================================================

total_params = sum(
    p.numel()
    for p in test_model.parameters()
)

print(
    f"📊 Parameters: "
    f"{total_params:,} "
    f"({total_params / 1e6:.2f}M)"
)


# ============================================================
# LOAD CHECKPOINT
# ============================================================

print(
    "\n🔄 Loading best_model.pt..."
)

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=TEST_DEVICE
)

if not isinstance(checkpoint, dict):
    raise TypeError(
        "❌ Unexpected checkpoint format."
    )

print("\n🔍 Checkpoint information")

print("Checkpoint keys:")
for key in checkpoint.keys():
    print(f"   • {key}")

if "model_state_dict" not in checkpoint:
    raise KeyError(
        "❌ 'model_state_dict' not found in checkpoint."
    )

test_model.load_state_dict(
    checkpoint["model_state_dict"]
)

test_model.eval()

print(
    "\n✅ Best model loaded successfully."
)


# ============================================================
# CHECKPOINT VERIFICATION
# ============================================================

checkpoint_state = checkpoint[
    "model_state_dict"
]

model_state = test_model.state_dict()

checkpoint_keys = set(
    checkpoint_state.keys()
)

model_keys = set(
    model_state.keys()
)

missing_keys = (
    model_keys -
    checkpoint_keys
)

unexpected_keys = (
    checkpoint_keys -
    model_keys
)

shape_mismatches = []

for key in checkpoint_keys & model_keys:

    if (
        checkpoint_state[key].shape
        !=
        model_state[key].shape
    ):
        shape_mismatches.append(
            (
                key,
                checkpoint_state[key].shape,
                model_state[key].shape
            )
        )

print(
    "\n🔍 Checkpoint verification"
)

print(
    f"Checkpoint tensors: "
    f"{len(checkpoint_keys):,}"
)

print(
    f"Model tensors:      "
    f"{len(model_keys):,}"
)

if missing_keys:
    print("\n❌ Missing keys:")
    for key in sorted(missing_keys):
        print("   ", key)
else:
    print("✅ No missing keys.")

if unexpected_keys:
    print("\n⚠️ Unexpected keys:")
    for key in sorted(unexpected_keys):
        print("   ", key)
else:
    print("✅ No unexpected keys.")

if shape_mismatches:
    print("\n❌ Shape mismatches:")
    for (
        key,
        checkpoint_shape,
        model_shape
    ) in shape_mismatches:
        print(
            f"   {key}: "
            f"checkpoint={checkpoint_shape}, "
            f"model={model_shape}"
        )
else:
    print("✅ All tensor shapes match.")

if (
    not missing_keys
    and not unexpected_keys
    and not shape_mismatches
):
    print(
        "\n🎯 Checkpoint architecture "
        "matches the model perfectly."
    )
else:
    raise RuntimeError(
        "\n❌ Checkpoint is not fully compatible."
    )


# ============================================================
# GENERATION FUNCTION
# ============================================================

def independent_generate(
    model,
    tokenizer,
    prompt,
    max_seq_len=128,
    temperature=0.7,
    top_k=10,
    greedy=False,
    device="cuda",
    seed=42
):

    model.eval()

    prompt_ids = tokenizer.encode(
        prompt
    ).ids

    if len(prompt_ids) == 0:
        raise ValueError(
            "❌ Prompt produced zero tokens."
        )

    if len(prompt_ids) >= max_seq_len:
        raise ValueError(
            "❌ Prompt is already equal to "
            "or longer than max_seq_len."
        )

    inputs = torch.tensor(
        prompt_ids,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    rng = torch.Generator(
        device=device
    )

    rng.manual_seed(seed)

    with torch.no_grad():

        while inputs.shape[1] < max_seq_len:

            logits = model(
                inputs
            )

            next_logits = logits[
                :,
                -1,
                :
            ]

            if greedy:

                next_token = torch.argmax(
                    next_logits,
                    dim=-1
                )

            else:

                temperature = max(
                    temperature,
                    1e-5
                )

                probs = torch.softmax(
                    next_logits / temperature,
                    dim=-1
                )

                k = min(
                    top_k,
                    probs.shape[-1]
                )

                topk_probs, topk_indices = (
                    torch.topk(
                        probs,
                        k=k,
                        dim=-1
                    )
                )

                # Important:
                # torch.multinomial expects a valid probability
                # distribution. Re-normalize top-k probabilities.
                topk_probs = (
                    topk_probs
                    /
                    topk_probs.sum(
                        dim=-1,
                        keepdim=True
                    )
                )

                sampled = torch.multinomial(
                    topk_probs,
                    num_samples=1,
                    generator=rng
                )

                next_token = torch.gather(
                    topk_indices,
                    -1,
                    sampled
                ).squeeze(-1)

            inputs = torch.cat(
                [
                    inputs,
                    next_token.unsqueeze(-1)
                ],
                dim=-1
            )

    generated_ids = inputs[
        0,
        len(prompt_ids):
    ].tolist()

    return tokenizer.decode(
        generated_ids
    )


# ============================================================
# LAYERED GENERATION
# ============================================================

def layered_generate(
    model,
    tokenizer,
    prompt,
    text_layers,
    max_seq_len=128,
    temperature=0.7,
    top_k=10,
    greedy=False,
    device="cuda",
    seed=42,
    auto_tone_convert=True,
):
    """
    Full production-like pipeline:

        raw user input
              ↓
        spelling normalization
              ↓
        user tone detection
              ↓
        model generation
              ↓
        generated-response tone detection
              ↓
        optional tone conversion
    """

    preprocessed = text_layers.preprocess_prompt(
        prompt
    )

    normalized_prompt = preprocessed[
        "normalized"
    ]

    user_tone = preprocessed[
        "tone"
    ]["tone"]

    raw_response = independent_generate(
        model=model,
        tokenizer=tokenizer,
        prompt=normalized_prompt,
        max_seq_len=max_seq_len,
        temperature=temperature,
        top_k=top_k,
        greedy=greedy,
        device=device,
        seed=seed,
    )

    postprocessed = text_layers.postprocess_response(
        raw_response,
        target_tone=user_tone,
        auto_convert=auto_tone_convert,
    )

    return {
        "original_prompt": prompt,
        "normalized_prompt": normalized_prompt,
        "user_tone": preprocessed["tone"],
        "raw_response": raw_response,
        "raw_response_tone": postprocessed["detected_tone"],
        "final_response": postprocessed["converted"],
        "final_response_tone": postprocessed["converted_tone"],
    }


# ============================================================
# TEST PROMPT
# ============================================================

TEST_PROMPT = (
    "یادگیری ماشین (Machine Learning - ML)، "
    "یکی از زیرشاخه‌های هوش مصنوعی است که"
)


# ============================================================
# NEXT TOKEN PROBABILITY TEST
# ============================================================

print("\n")
print("=" * 100)
print("🔬 NEXT TOKEN PROBABILITY TEST")
print("=" * 100)

prompt = (
    "یادگیری ماشین (Machine Learning - ML)، "
    "یکی از زیرشاخه‌های هوش مصنوعی است که"
)

print("\n[Prompt]")
print(prompt)

ids = test_tokenizer.encode(
    prompt
).ids

print("\n[Prompt Token IDs]")
print(ids)

print(
    f"\nPrompt token count: "
    f"{len(ids)}"
)

x = torch.tensor(
    [ids],
    dtype=torch.long,
    device=TEST_DEVICE
)

with torch.no_grad():
    logits = test_model(x)

next_logits = logits[
    :,
    -1,
    :
]

probs = torch.softmax(
    next_logits,
    dim=-1
)

top_probs, top_ids = torch.topk(
    probs,
    k=min(20, probs.shape[-1]),
    dim=-1
)

print("\n")
print("-" * 100)
print("TOP-20 NEXT TOKEN PREDICTIONS")
print("-" * 100)
print(
    f"{'Rank':>4} "
    f"{'ID':>6} "
    f"{'Probability':>14} "
    f"{'Token':>30}"
)
print("-" * 100)

for rank, (p, idx) in enumerate(
    zip(
        top_probs[0],
        top_ids[0]
    ),
    start=1
):
    token = test_tokenizer.decode(
        [idx.item()]
    )

    print(
        f"{rank:4d} "
        f"{idx.item():6d} "
        f"{p.item():14.8f} "
        f"{repr(token):>30}"
    )


# ============================================================
# EXPECTED CONTINUATION TEST
# ============================================================

print("\n")
print("=" * 100)
print("🎯 EXPECTED NEXT TOKEN TEST")
print("=" * 100)

expected = (
    " به سیستم‌ها اجازه می‌دهد"
)

expected_ids = test_tokenizer.encode(
    expected
).ids

print("\n[Expected continuation]")
print(repr(expected))

print("\n[Expected token IDs]")
print(expected_ids)

print("\n[Expected tokens]")

for position, idx in enumerate(
    expected_ids,
    start=1
):
    token = test_tokenizer.decode(
        [idx]
    )

    print(
        f"{position:3d}. "
        f"ID={idx:5d} "
        f"Token={repr(token)}"
    )

if len(expected_ids) > 0:

    expected_next_id = expected_ids[0]

    expected_prob = (
        probs[
            0,
            expected_next_id
        ].item()
    )

    print("\n")
    print("-" * 100)
    print("ACTUAL EXPECTED NEXT TOKEN")
    print("-" * 100)

    print(
        "Expected token ID:",
        expected_next_id
    )

    print(
        "Expected token:",
        repr(
            test_tokenizer.decode(
                [expected_next_id]
            )
        )
    )

    print(
        "Probability:",
        f"{expected_prob:.10f}"
    )

    sorted_ids = torch.argsort(
        probs[0],
        descending=True
    )

    expected_rank_tensor = torch.where(
        sorted_ids == expected_next_id
    )[0]

    if len(expected_rank_tensor) > 0:
        expected_rank = (
            expected_rank_tensor[0].item()
            + 1
        )

        print(
            "Rank:",
            expected_rank
        )
    else:
        print("Rank: Not found")


# ============================================================
# GREEDY DECODING
# ============================================================

print("\n")
print("=" * 100)
print("🧠 GREEDY DECODING")
print("=" * 100)

greedy_output = independent_generate(
    model=test_model,
    tokenizer=test_tokenizer,
    prompt=TEST_PROMPT,
    max_seq_len=128,
    greedy=True,
    device=TEST_DEVICE
)

print("\n[Prompt]")
print(TEST_PROMPT)

print("\n[Generated continuation]")
print(greedy_output)


# ============================================================
# TOP-K SAMPLING
# ============================================================

print("\n")
print("=" * 100)
print("🎲 TOP-K SAMPLING")
print("=" * 100)

for sample_number in range(
    1,
    4
):

    output = independent_generate(
        model=test_model,
        tokenizer=test_tokenizer,
        prompt=TEST_PROMPT,
        max_seq_len=128,
        temperature=0.7,
        top_k=10,
        greedy=False,
        device=TEST_DEVICE,
        seed=42 + sample_number
    )

    print(
        f"\n[Sample {sample_number}]"
    )

    print(output)


# ============================================================
# LAYERED GENERATION TEST
# ============================================================

print("\n")
print("=" * 100)
print("🤖 LAYERED MODEL + TEXT LAYERS TEST")
print("=" * 100)

layered_prompt = (
    "می تونی یادگیری ماشین رو برام توضیح بدی؟"
)

result = layered_generate(
    model=test_model,
    tokenizer=test_tokenizer,
    prompt=layered_prompt,
    text_layers=text_layers,
    max_seq_len=96,
    temperature=0.7,
    top_k=10,
    greedy=False,
    device=TEST_DEVICE,
    seed=123,
    auto_tone_convert=True,
)

print("\n[Original user prompt]")
print(result["original_prompt"])

print("\n[Normalized user prompt]")
print(result["normalized_prompt"])

print("\n[User tone]")
print(result["user_tone"])

print("\n[Raw model response]")
print(result["raw_response"])

print("\n[Raw model response tone]")
print(result["raw_response_tone"])

print("\n[Final response]")
print(result["final_response"])

print("\n[Final response tone]")
print(result["final_response_tone"])


# ============================================================
# TOKENIZER TEST
# ============================================================

print("\n")
print("=" * 100)
print("✅ TOKENIZER TEST")
print("=" * 100)

texts = [
    "یادگیری ماشین",
    (
        "یادگیری ماشین "
        "(Machine Learning - ML)، "
        "یکی از زیرشاخه‌های هوش مصنوعی است که"
    )
]

for text in texts:

    enc = test_tokenizer.encode(
        text
    )

    print("\nTEXT:")
    print(text)

    print("IDS:")
    print(enc.ids)

    print("TOKENS:")
    print(enc.tokens)


# ============================================================
# TOKENIZER ROUND-TRIP TEST
# ============================================================

print("\n")
print("=" * 80)
print("🔬 TOKENIZER ROUND-TRIP TEST")
print("=" * 80)

for text in texts:

    enc = test_tokenizer.encode(
        text
    )

    decoded = test_tokenizer.decode(
        enc.ids
    )

    print("\nOriginal:")
    print(repr(text))

    print("Decoded:")
    print(repr(decoded))

    print(
        "MATCH:",
        text == decoded
    )


# ============================================================
# FULL TARGET TEST
# ============================================================

target = """یادگیری ماشین (Machine Learning - ML)، یکی از زیرشاخه‌های هوش مصنوعی است که به سیستم‌ها اجازه می‌دهد بدون برنامه‌ریزی صریح، از داده‌ها یاد بگیرند و بهبود یابند.

الگوریتم‌های یادگیری ماشین به سه دسته‌ی اصلی تقسیم می‌شوند:

- **یادگیری نظارت‌شده (Supervised Learning)**
- **یادگیری بدون نظارت (Unsupervised Learning)**
- **یادگیری تقویتی (Reinforcement Learning)**"""

enc = test_tokenizer.encode(
    target
)

decoded = test_tokenizer.decode(
    enc.ids
)

print("\n")
print("=" * 100)
print("📚 FULL TARGET TOKENIZER TEST")
print("=" * 100)

print("\nOriginal:")
print(repr(target))

print("\nDecoded:")
print(repr(decoded))

print(
    "\nMATCH:",
    target == decoded
)

print(
    "Token count:",
    len(enc.ids)
)


# ============================================================
# FINAL
# ============================================================

print("\n")
print("=" * 100)
print("✅ ALL TESTS COMPLETED")
print("=" * 100)
print(
    "Text layer directory:",
    TEXT_LAYERS_DIR
)

=== MODAL PATH DEBUG ===

/data:
['dataset', 'logs', 'parquet', 'raw', 'tokenizer']

/raw:
NOT FOUND

/mnt:
[]

/workspace:
NOT FOUND
📝 PERSIAN TEXT LAYERS
✅ halfspace.json loaded: /data/raw/persian_text_layers/halfspace.json
   Normalization rules: 328
   Correct forms:       322
✅ casual.json loaded:   /data/raw/persian_text_layers/casual.json
   Formal→casual rules: 22


🧪 TEST 1 — INPUT NORMALIZATION

Original :
'بالغ بر'
Normalized:
'بالغ\u200cبر'
RESULT: ✅ CONVERTED

Original :
'حمل و نقل'
Normalized:
'حمل\u200cونقل'
RESULT: ✅ CONVERTED

Original :
'زنده گی'
Normalized:
'زنده گی'
RESULT: ⚠️ NO REGISTERED CHANGE

Original :
'می رود'
Normalized:
'می رود'
RESULT: ⚠️ NO REGISTERED CHANGE

Original :
'می شود'
Normalized:
'می شود'
RESULT: ⚠️ NO REGISTERED CHANGE

Original :
'می تواند'
Normalized:
'می تواند'
RESULT: ⚠️ NO REGISTERED CHANGE

Original :
'به کار گیری'
Normalized:
'به\u200cکار گیری'
RESULT: ✅ CONVERTED

Normalization tests with a change: 3/7


🧪 TEST 2 — USER INPUT TONE DET